In [0]:
file_path = "abfss://rawdata@azurestoragedbk4.dfs.core.windows.net/Dmart/dmart_dailysales.csv"
file_content = spark.read.format("csv").option("header", "true").load(file_path)
display(file_content.limit(10))

Name,Brand,Price,DiscountedPrice,Category,SubCategory,Quantity,Description,BreadCrumbs
Premia Badam (Almonds),Premia,451,329,Grocery,Grocery/Dry Fruits,500 gm,India,Grocery > Grocery/Dry Fruits
Premia Badam (Almonds),Premia,109,85,Grocery,Grocery/Dry Fruits,100 gm,India,Grocery > Grocery/Dry Fruits
Premia Badam (Almonds),Premia,202,175,Grocery,Grocery/Dry Fruits,200 gm,India,Grocery > Grocery/Dry Fruits
Nutraj California Almonds (Badam),Nutraj,599,349,Grocery,Dry Fruits,500 gm,USA,Grocery > Dry Fruits
Nutraj California Almonds (Badam),Nutraj,1549,659,Grocery,Dry Fruits,1 kg,USA,Grocery > Dry Fruits
Chana Dal,null,49,42,Grocery,Dals,500 gm,India,Grocery > Dals
Chana Dal,null,96,80,Grocery,Dals,1 kg,India,Grocery > Dals
Rajma White,null,112,102,Grocery,Pulses,500 gm,India,Grocery > Pulses
Rajma Kashmiri Red,null,101,81,Grocery,Pulses,500 gm,India,Grocery > Pulses
Premia Mamra Badam,Premia,634,488,Grocery,Dry Fruits,200 gm,India,Grocery > Dry Fruits


**Bronze Layer**

In [0]:
# Bronze: Raw data
bronze_data = file_content
bronze_path = "abfss://medallionarch@azurestoragedbk4.dfs.core.windows.net/bronze/dmart_dailysales"
bronze_data.write.mode("overwrite").format("csv").save(bronze_path)

**Silver Layer**

In [0]:
# Silver: Cleaned data (remove duplicates, handle nulls, correct types, filter invalid rows)
from pyspark.sql.functions import col, regexp_extract

# Define columns and their target types
numeric_columns = ["Price", "DiscountedPrice", "Quantity"]

# Filter rows where numeric columns are valid numbers
valid_df = bronze_data
for col_name in numeric_columns:
    valid_df = valid_df.filter(regexp_extract(col(col_name), '^[0-9]+([0-9]+)?$', 0) != "")

# Cast numeric columns to appropriate types
valid_df = valid_df.withColumn("Price", col("Price").cast("double")) \
                   .withColumn("DiscountedPrice", col("DiscountedPrice").cast("double")) \
                   .withColumn("Quantity", col("Quantity").cast("integer"))

# Remove duplicates and nulls
silver_df = valid_df.dropDuplicates().na.drop()

silver_path = "abfss://medallionarch@azurestoragedbk4.dfs.core.windows.net/silver/dmart_dailysales"
silver_df.write.mode("overwrite").format("csv").save(silver_path)

**Gold Layer**

In [0]:
# Gold: Aggregated data (example: total sales per store per day)
gold_df = silver_df.groupBy("Name", "Category").agg({"Price": "sum"}).withColumnRenamed("sum(Price)", "TotalSales")
gold_path = "abfss://medallionarch@azurestoragedbk4.dfs.core.windows.net/gold/dmart_dailysales"
gold_df.write.mode("overwrite").format("csv").save(gold_path)